In [15]:
import galsim
import jax.numpy as jnp
import jax_galsim as xgalsim
from jax import random
from jax._src.prng import PRNGKeyArray
from jax.typing import ArrayLike
from jax_galsim import GSParams

from functools import partial

import jax 

In [16]:
def draw_gaussian(
    *,
    f: float,
    hlr: float,
    e1: float,
    e2: float,
    x: float,  # pixels
    y: float,
    slen: int,
    fft_size: int,  # rule of thumb: at least 4 times `slen`
    psf_fwhm: float = 0.8,
    pixel_scale: float = 0.2,
):
    gsparams = GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)

    gal = xgalsim.Gaussian(flux=f, half_light_radius=hlr)
    gal = gal.shear(g1=e1, g2=e2)

    psf = xgalsim.Gaussian(flux=1.0, fwhm=0.8)
    gal_conv = xgalsim.Convolve([gal, psf]).withGSParams(gsparams)
    image = gal_conv.drawImage(nx=slen, ny=slen, scale=pixel_scale, offset=(x, y))
    return image.array

In [17]:
def draw_gaussian_moffat(
    *,
    f: float,
    hlr: float,
    e1: float,
    e2: float,
    x: float,  # pixels
    y: float,
    slen: int,
    fft_size: int,  # rule of thumb: at least 4 times `slen`
    psf_fwhm: float = 0.8,
    pixel_scale: float = 0.2,
):
    gsparams = GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)

    gal = xgalsim.Gaussian(flux=f, half_light_radius=hlr)
    gal = gal.shear(g1=e1, g2=e2)

    psf = xgalsim.Moffat(flux=1.0, scale_radius=0.8, beta=2.0)
    gal_conv = xgalsim.Convolve([gal, psf]).withGSParams(gsparams)
    image = gal_conv.drawImage(nx=slen, ny=slen, scale=pixel_scale, offset=(x, y))
    return image.array

In [18]:
_func1 = jax.jit(partial(draw_gaussian, slen=63, fft_size=256))
_ = _func1(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0)

In [19]:
%%timeit
_func1(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0)

228 μs ± 4.69 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
_func2 = jax.jit(partial(draw_gaussian_moffat, slen=63, fft_size=256))
_ = _func2(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0)

In [21]:
%%timeit
_func2(f=1.0, hlr=1.0, e1=0.2, e2=0.2, x=0., y=0.0) 

2.16 ms ± 20.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
